# Phần 2: Ứng dụng Hồi quy trên Dữ liệu Thực tế

## Luồng công việc
1. **EDA**: Phân tích dữ liệu khám phá (Nam)
2. **Tiền xử lý**: Sử dụng `DataPipeline` (Khiêm)
3. **Huấn luyện mô hình**: OLS, Ridge/Lasso (Minh)
4. **Đánh giá**: Kiểm thử và phân tích lỗi (Kiên)

# ĐỒ ÁN 2: DATA FITTING VÀ PHƯƠNG PHÁP OLS
## PHẦN 2: ỨNG DỤNG DATA FITTING VÀO DỮ LIỆU THỰC TẾ

**Thành viên thực hiện nhiệm vụ Hồi quy & Xây dựng mô hình:** Lê Quang Minh 

**Mục tiêu:** Tiến hành chia tập Train/Test, áp dụng Pipeline làm sạch dữ liệu và xây dựng 3 mô hình hồi quy (OLS cơ bản, OLS chọn biến Stepwise Backward, và Ridge Regression) để dự báo doanh thu toàn cầu (`Global_Sales`) của bộ dữ liệu Video Games Sales

In [43]:
# Khai báo và import các thư viện cần thiết cho luồng xử lý mô hình
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV

# Import trực tiếp class DataPipeline được thiết kế bởi thành viên quản lý dữ liệu
from data_pipeline import DataPipeline

print("[*] Đã tải thành công các thư viện và pipeline tiền xử lý.")

[*] Đã tải thành công các thư viện và pipeline tiền xử lý.


### 2.3 Xây Dựng và Đánh Giá Mô Hình
#### 2.3.1 Quy trình xây dựng mô hình

Dựa theo quy trình chuẩn của đồ án, dữ liệu sau quá trình EDA sẽ được đưa vào Pipeline để tiền xử lý. Tuy nhiên, để ngăn chặn hiện tượng rò rỉ dữ liệu (Data Leakage), tập dữ liệu thô bắt buộc phải được phân tách thành hai tập Train/Test theo tỷ lệ 80:20 trước khi thực hiện bước tiền xử lý.

In [44]:
# 1. Đọc dữ liệu từ thư mục quản lý data của nhóm
# Do Notebook chạy trực tiếp trong thư mục part2, đường dẫn tương đối sẽ tính từ part2/
csv_path = os.path.join("data", "video_games_sales.csv")
df = pd.read_csv(csv_path)

# Đảm bảo loại bỏ các bản ghi không có biến mục tiêu (nếu có)
df = df.dropna(subset=['Global_Sales']) 

# Tách tập thuộc tính độc lập X và biến mục tiêu liên tục y
X = df.drop(columns=['Global_Sales'])
y = df['Global_Sales']

# Chia tập dữ liệu theo tỷ lệ chuẩn 8-2, thiết lập seed cố định để kết quả có thể tái lập
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

print(f"Kích thước dữ liệu Train thô: {X_train_raw.shape}")
print(f"Kích thước dữ liệu Test thô: {X_test_raw.shape}")

Kích thước dữ liệu Train thô: (13375, 15)
Kích thước dữ liệu Test thô: (3344, 15)


#### 2.3.2 Các Mô Hình Cần Thử Nghiệm

Nhóm tiến hành xây dựng và thử nghiệm 3 mô hình theo yêu cầu bắt buộc của đồ án:
1. **OLS cơ bản (Full Model):** Hồi quy với tất cả các biến sau tiền xử lý.
2. **OLS chọn biến (Short Model):** Sử dụng phương pháp Stepwise Backward để loại bỏ các biến không có ý nghĩa thống kê (dựa trên p-value > 0.05).
3. **Ridge Regression:** Sử dụng kỹ thuật Regularization và chọn siêu tham số lambda (alpha) thông qua K-fold Cross Validation (k=5).

In [45]:
# Khởi tạo pipeline lưu giữ top 30 nhà phát hành có doanh số cao nhất
pipeline = DataPipeline(top_n_publishers=30)

# Thực hiện học thông số và biến đổi cấu trúc trên tập huấn luyện
X_train_clean = pipeline.fit_transform(X_train_raw)

# Áp dụng các thông số cũ để biến đổi cấu trúc trên tập kiểm thử
X_test_clean = pipeline.transform(X_test_raw)

# Ép kiểu dữ liệu sang dạng float để tương thích tuyệt đối với các ma trận của statsmodels
X_train_clean = X_train_clean.astype(float)
X_test_clean = X_test_clean.astype(float)
y_train = y_train.astype(float)
y_test = y_test.astype(float)

print(f"Kích thước ma trận đặc trưng Train sau xử lý: {X_train_clean.shape}")
print(f"Kích thước ma trận đặc trưng Test sau xử lý: {X_test_clean.shape}")

Kích thước ma trận đặc trưng Train sau xử lý: (13375, 79)
Kích thước ma trận đặc trưng Test sau xử lý: (3344, 79)


#### 2.3.3 Thử nghiệm các mô hình hồi quy tuyến tính bắt buộc

##### Mô hình 1: Hồi quy tuyến tính OLS cơ bản (Full Model) 
Mô hình này bao gồm toàn bộ các thuộc tính thu được sau quá trình tiền xử lý, đóng vai trò là mô hình nền tảng để so sánh và đánh giá mức độ cải thiện hiệu năng thống kê.

In [46]:
# Bổ sung cột hằng số hằng định (const = 1) phục vụ tính toán hệ số chặn Intercept
X_train_sm = sm.add_constant(X_train_clean)
X_test_sm = sm.add_constant(X_test_clean)

# Xây dựng và khớp mô hình OLS toàn diện
full_model = sm.OLS(y_train, X_train_sm).fit()

# Hiển thị bảng kết quả phân tích hệ số
print(full_model.summary())

                            OLS Regression Results                            
Dep. Variable:           Global_Sales   R-squared:                       0.237
Model:                            OLS   Adj. R-squared:                  0.232
Method:                 Least Squares   F-statistic:                     54.24
Date:                Mon, 18 May 2026   Prob (F-statistic):               0.00
Time:                        10:56:32   Log-Likelihood:                -21682.
No. Observations:               13375   AIC:                         4.352e+04
Df Residuals:                   13298   BIC:                         4.410e+04
Df Model:                          76                                         
Covariance Type:            nonrobust                                         
                                                       coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------

##### Mô hình 2: OLS chọn lọc biến đặc trưng (Stepwise Backward Elimination) 
Dựa vào kết quả chỉ báo hệ số phóng đại phương sai (VIF) hoặc kiểm định ý nghĩa $p\text{-value}$ từ Full Model, ta nhận thấy xuất hiện hiện tượng đa cộng tuyến cao do số lượng biến giả tăng mạnh. Thuật toán loại bỏ ngược (Backward Elimination) sẽ lặp lại chu kỳ xây dựng mô hình, tìm và loại bỏ thuộc tính có $p\text{-value} > 0.05$ lớn nhất tại mỗi bước cho đến khi thu được tập biến tối ưu.

In [47]:
def backward_elimination(X, y, significance_level=0.05):
    features = X.columns.tolist()
    step = 1
    while len(features) > 0:
        X_with_const = sm.add_constant(X[features])
        model = sm.OLS(y, X_with_const).fit()
        
        # Trích xuất p-values, bỏ qua biến const để bảo toàn intercept
        p_values = model.pvalues.drop('const', errors='ignore')
        max_p_value = p_values.max()
        
        if max_p_value > significance_level:
            excluded_feature = p_values.idxmax()
            features.remove(excluded_feature)
            print(f"Bước {step}: Loại bỏ {excluded_feature:<30} | p-value = {max_p_value:.4f}")
            step += 1
        else:
            break
            
    final_model = sm.OLS(y, sm.add_constant(X[features])).fit()
    return final_model, features

# Tiến hành chạy thuật toán tinh gọn đặc trưng
short_model, selected_features = backward_elimination(X_train_clean, y_train)

print("\n--- KẾT QUẢ RÚT GỌN MÔ HÌNH SHORT MODEL ---")
print(f"Số lượng đặc trưng được giữ lại có ý nghĩa thống kê: {len(selected_features)} / {X_train_clean.shape[1]}")
print(short_model.summary())

Bước 1: Loại bỏ Publisher_Nintendo             | p-value = 0.9893
Bước 2: Loại bỏ Genre_Unknown                  | p-value = 0.9875
Bước 3: Loại bỏ Platform_NES                   | p-value = 0.9972
Bước 4: Loại bỏ Genre_Platform                 | p-value = 0.3074
Bước 5: Loại bỏ Platform_GB                    | p-value = 0.2828
Bước 6: Loại bỏ Platform_PCFX                  | p-value = 0.1762
Bước 7: Loại bỏ Platform_TG16                  | p-value = 0.1629
Bước 8: Loại bỏ Platform_GG                    | p-value = 0.0995
Bước 9: Loại bỏ Platform_3DO                   | p-value = 0.0718

--- KẾT QUẢ RÚT GỌN MÔ HÌNH SHORT MODEL ---
Số lượng đặc trưng được giữ lại có ý nghĩa thống kê: 70 / 79
                            OLS Regression Results                            
Dep. Variable:           Global_Sales   R-squared:                       0.236
Model:                            OLS   Adj. R-squared:                  0.232
Method:                 Least Squares   F-statistic:           

##### Mô hình 3: Hồi quy Ridge với kỹ thuật hiệu chỉnh Regularization ($L_2$) 
Mô hình Ridge bổ sung thành phần phạt bình phương độ lớn hệ số hồi quy nhắm tới việc triệt tiêu ảnh hưởng tiêu cực của hiện tượng đa cộng tuyến nghiêm trọng mà không cần loại bỏ biến. Việc tìm kiếm siêu tham số phạt tối ưu ($\alpha$ hoặc $\lambda$) được thực hiện thông qua cơ chế K-fold Cross Validation ($k=5$) trực tiếp trên tập huấn luyện.

In [48]:
# Danh sách tập hợp các giá trị alpha thực nghiệm cần sàng lọc
alphas_to_test = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

# Khởi tạo mô hình kiểm chứng chéo Ridge với k-fold = 5
ridge_cv_model = RidgeCV(alphas=alphas_to_test, cv=5)
ridge_cv_model.fit(X_train_clean, y_train)

print(f"[*] Siêu tham số tối ưu lựa chọn qua Cross Validation: alpha = {ridge_cv_model.alpha_}") 
print(f"[*] Chỉ số R^2 đạt được trên tập huấn luyện: {ridge_cv_model.score(X_train_clean, y_train):.4f}")

[*] Siêu tham số tối ưu lựa chọn qua Cross Validation: alpha = 10.0
[*] Chỉ số R^2 đạt được trên tập huấn luyện: 0.2363


#### 2.3.4 Xuất dữ liệu dự báo phục vụ công tác đánh giá sai số 

Để phục vụ phân đoạn tiếp theo của kỹ sư kiểm định mô hình, các mảng giá trị dự báo từ 3 cấu hình thuật toán trên tập Test độc lập được trích xuất đồng bộ ra file lưu trữ cấu trúc `.csv` nhằm phục vụ việc phân tích phần dư và đo đạc các thang điểm RMSE, MAE.

In [49]:
# Khởi tạo bảng dữ liệu lưu trữ kết quả đầu ra
results_df = pd.DataFrame({'Actual_Sales': y_test})

# 1. Trích xuất dự báo từ Full Model OLS
results_df['Pred_Full_Model'] = full_model.predict(X_test_sm)

# 2. Trích xuất dự báo từ Short Model OLS (Chỉ dùng các đặc trưng chọn lọc)
X_test_short = sm.add_constant(X_test_clean[selected_features])
results_df['Pred_Short_Model'] = short_model.predict(X_test_short)

# 3. Trích xuất dự báo từ Ridge Regression
results_df['Pred_Ridge'] = ridge_cv_model.predict(X_test_clean)

# Ghi dữ liệu kết quả ra các file lưu trữ cấu trúc trong phân vùng data/
features_out_path = os.path.join("data", "selected_features.csv")
predictions_out_path = os.path.join("data", "model_predictions.csv")

pd.Series(selected_features).to_csv(features_out_path, index=False)
results_df.to_csv(predictions_out_path, index=False)

print(f"[+] Hoàn tất lưu danh sách biến chọn lọc tại: {features_out_path}")
print(f"[+] Hoàn tất lưu mảng dữ liệu dự đoán tại: {predictions_out_path}")

[+] Hoàn tất lưu danh sách biến chọn lọc tại: data\selected_features.csv
[+] Hoàn tất lưu mảng dữ liệu dự đoán tại: data\model_predictions.csv


In [50]:
import sys
sys.path.append('..')
import pandas as pd
from data_pipeline import DataPipeline

# Tải dữ liệu
df = pd.read_csv('data/video_games_sales.csv')
print("Shape của dữ liệu:", df.shape)
df.head(5)

Shape của dữ liệu: (16719, 16)


,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8,322.0,Nintendo,E
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8,192.0,Nintendo,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,NaN,NaN
